# HubbardBath 07 — LCU block encoding

Milestone 05 gave us a Pauli decomposition of the two-site Fermi–Hubbard Hamiltonian,

$$H=\sum_j \alpha_j P_j.$$

Milestone 07 packages that decomposition into a larger unitary whose all-zero-ancilla block is exactly $H/\alpha$, where

$$\alpha=\sum_j |\alpha_j|.$$

This is the matrix-access primitive we need before QSVT becomes meaningful.

In [ ]:
from pathlib import Path
import sys
import numpy as np

candidates=[Path.cwd(),Path.cwd().parent,Path.cwd()/'HubbardBath']
for c in candidates:
    if (c/'src'/'block_encoding.py').exists():
        repo_root=c.resolve(); break
else:
    raise FileNotFoundError('Run from the HubbardBath repository')
if str(repo_root) not in sys.path:
    sys.path.insert(0,str(repo_root))

from src.block_encoding import (
    block_encoding_error, block_encoding_unitary,
    build_lcu_data, prepare_target_state, projected_action,
    top_left_system_block, unitarity_error,
)
from src.qubit_mapping import (
    reconstruct_from_pauli_terms, remove_zero_terms,
    two_site_hubbard_pauli_terms,
)
from src.simulation import computational_basis_state


## 1. Start from the Pauli Hamiltonian

For the reference point $U/t=4$ and $\mu=0$, the four-qubit Hamiltonian has 11 nonzero Pauli terms.

In [ ]:
terms=remove_zero_terms(two_site_hubbard_pauli_terms(t=1.0,U=4.0,mu=0.0))
data=build_lcu_data(terms)

print('Pauli terms:',len(data.labels))
print('LCU alpha:',data.alpha)
print('System qubits:',data.system_qubits)
print('Ancilla qubits:',data.ancilla_qubits)
print('Total qubits in dense reference:',data.system_qubits+data.ancilla_qubits)


## 2. PREPARE

PREPARE creates

$$|G\rangle=\sum_j\sqrt{|\alpha_j|/\alpha}\,|j\rangle.$$

The probability weights encode coefficient magnitudes.

In [ ]:
target=prepare_target_state(data)
for j,label in enumerate(data.labels):
    print(f'{j:2d}  {label}  amplitude={target[j].real:.6f}  probability={data.probabilities[j]:.6f}')

print('Total probability:',np.sum(np.abs(target)**2))


## 3. SELECT and the full block encoding

SELECT applies the Pauli term conditioned on the ancilla label, including the sign or phase of each coefficient. We then form

$$U_H=(A^\dagger\otimes I)\,\mathrm{SELECT}\,(A\otimes I),$$

where $A$ is PREPARE.

In [ ]:
U_block,data=block_encoding_unitary(terms)
H=reconstruct_from_pauli_terms(terms)
encoded=top_left_system_block(U_block,data.system_dimension)

print('Full unitary dimension:',U_block.shape)
print(f'Unitarity error: {unitarity_error(U_block):.3e}')
print(f'Top-left block error: {block_encoding_error(terms):.3e}')
print(f'||H/alpha||_2: {np.linalg.norm(H/data.alpha,ord=2):.6f}')

assert unitarity_error(U_block)<1e-12
assert block_encoding_error(terms)<1e-12


## 4. Verify the defining identity directly

The mathematical statement of the block encoding is

$$(\langle0|\otimes I)U_H(|0\rangle\otimes I)=H/\alpha.$$


In [ ]:
difference=encoded-H/data.alpha
print('Maximum absolute matrix entry error:',np.max(np.abs(difference)))
np.testing.assert_allclose(encoded,H/data.alpha,atol=1e-12)


## 5. What happens to an actual state?

Starting from $|0\rangle_a|\psi\rangle_s$, applying the block encoding and projecting the ancilla back onto zero produces the unnormalized state

$$\frac{H}{\alpha}|\psi\rangle.$$

We verify this for the same doublon state used in the thermalisation and Hamiltonian-simulation milestones.

In [ ]:
psi=computational_basis_state('1100')
projected,p_success=projected_action(U_block,data,psi)
expected=(H/data.alpha)@psi

print(f'Projected-action error: {np.linalg.norm(projected-expected):.3e}')
print(f'Zero-ancilla probability: {p_success:.6f}')
np.testing.assert_allclose(projected,expected,atol=1e-12)


## What this milestone establishes

We now have an explicit and numerically verified block encoding of the Hubbard Hamiltonian. The dense 8-qubit reference unitary is **not** presented as an optimized fault-tolerant circuit; it is a transparent correctness construction.

**Next milestone:** use the normalized spectral variable provided by the block encoding to approximate a thermal function such as $e^{-\beta H/2}$ with bounded polynomials, then connect that approximation to the QSVT framework.